# 04 · Conectando el agente a un servidor MCP

Juntamos los notebooks 02 y 03: en vez de pasarle al agente una función Python local,
le vamos a pasar un **servidor MCP completo** — las mismas tools genéricas
(`sumar`, `crear_nota`, `listar_notas`, `completar_nota`) que ya construimos, pero esta
vez consumidas a través del protocolo, no importadas directamente.

Esto es exactamente el patrón de `jub-agent/agent/jub_agent.py::build_agent()`.

Requisitos: los mismos que en 02 y 03 (`fastmcp`, `agent-framework-ollama --pre`,
Ollama corriendo con el modelo descargado).


In [2]:
import os
import socket
import subprocess
import sys
import time
from pathlib import Path

MCP_SERVER_DIR = Path("..") / "mcp_server"
MCP_PORT = 8101  # puerto propio para este notebook

env = os.environ.copy()
env["MCP_PORT"] = str(MCP_PORT)

server_process = subprocess.Popen(
    [sys.executable, "server.py"],
    cwd=MCP_SERVER_DIR,
    env=env,
)


def esperar_puerto(host, port, timeout=30):
    inicio = time.time()
    while time.time() - inicio < timeout:
        try:
            with socket.create_connection((host, port), timeout=1):
                return True
        except OSError:
            time.sleep(0.5)
    raise TimeoutError(f"El servidor no abrió el puerto {port} a tiempo")


esperar_puerto("localhost", MCP_PORT)
print(f"Servidor MCP arriba en http://localhost:{MCP_PORT}/mcp")




╭──────────────────────────────────────────────────────────────────────────────╮
│                                                                              │
│                                                                              │
│                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │
│                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │
│                                                                              │
│                                                                              │
│                                                                              │
│                                FastMCP 3.4.7                                 │
│                            https://gofastmcp.com                             │
│                                                                              │
│                  🖥  Server:      tutorial-mcp, 3.4.7                         │
│                  🚀 Deplo

Servidor MCP arriba en http://localhost:8101/mcp


## `MCPStreamableHTTPTool`: el puente entre agent-framework y MCP

`agent-framework` trae tres tipos de "tool MCP" según el transporte del servidor
(`MCPStdioTool`, `MCPStreamableHTTPTool`, `MCPWebsocketTool`). Como nuestro servidor
usa `streamable-http`, usamos `MCPStreamableHTTPTool` — se comporta como una tool más
de cara al `Agent`, pero por dentro mantiene una `ClientSession` MCP real (la misma que
usamos a mano en los notebooks 01 y 02) y expone **todas** las tools del servidor de
una sola vez, sin tener que declararlas una por una.

Este es el mismo objeto y el mismo patrón que usa `jub_agent.py::build_agent()`:

```python
tools=MCPStreamableHTTPTool(name="jub-mcp", url="http://jub-mcp:8000/mcp")
```


In [3]:
from agent_framework import MCPStreamableHTTPTool
from agent_framework.ollama import OllamaChatClient

OLLAMA_URL = "http://localhost:11434"
OLLAMA_MODEL = "qwen2.5:1.5b"

mcp_tool = MCPStreamableHTTPTool(
    name="tutorial-mcp",
    url=f"http://localhost:{MCP_PORT}/mcp",
)

agent = OllamaChatClient(host=OLLAMA_URL, model=OLLAMA_MODEL).as_agent(
    name="TutorAgent",
    instructions=(
        "Eres un asistente que responde usando las herramientas MCP disponibles "
        "(calculadora y notas). Usa siempre una tool cuando la pregunta lo requiera "
        "en vez de inventar la respuesta. Responde en español, breve y directo."
    ),
    tools=mcp_tool,
)

async with agent:
    r1 = await agent.run("Crea una nota que diga 'comprar café' y confírmame que quedó guardada")
    print("Respuesta 1:", r1.text)

    r2 = await agent.run("¿Cuánto es 23 multiplicado por 8?")
    print("Respuesta 2:", r2.text)

    r3 = await agent.run("Lista mis notas pendientes")
    print("Respuesta 3:", r3.text)




╭──────────────────────────────────────────────────────────────────────────────╮
│                                                                              │
│                                                                              │
│                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │
│                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │
│                                                                              │
│                                                                              │
│                                                                              │
│                                FastMCP 3.4.7                                 │
│                            https://gofastmcp.com                             │
│                                                                              │
│                  🖥  Server:      tutorial-mcp, 3.4.7                         │
│                  🚀 Deplo

INFO:     127.0.0.1:39548 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:39564 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:39580 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:39594 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:39600 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:39602 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:39604 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:60070 - "POST /mcp HTTP/1.1" 200 OK
Respuesta 1: La nota "comprar café" se ha creado correctamente.
INFO:     127.0.0.1:60074 - "POST /mcp HTTP/1.1" 200 OK
Respuesta 2: El resultado de multiplicar 23 por 8 es 184.
INFO:     127.0.0.1:60078 - "POST /mcp HTTP/1.1" 200 OK
Respuesta 3: Tienes una nota pendiente: comprar café. No la has completado todavía.
INFO:     127.0.0.1:55418 - "DELETE /mcp HTTP/1.1" 200 OK


## Comparación notebook 03 vs. notebook 04

| | Notebook 03 | Notebook 04 |
|---|---|---|
| `tools=` | una función Python (`calcular_propina`) | un `MCPStreamableHTTPTool` apuntando a un servidor externo |
| Dónde vive la tool | en el proceso del notebook | en otro proceso (`mcp_server/server.py`), potencialmente otra máquina |
| Cuántas tools expone | una, declarada explícitamente | todas las que el servidor registre — el agente las descubre solo |
| Código de `Agent`/`.run()` | idéntico | idéntico |

Esa última fila es el punto: **`agent-framework` no distingue entre una tool local y
una tool MCP** una vez que están dentro de `tools=`. Por eso conectar un agente a un
servidor MCP nuevo (por ejemplo, uno con las tools de otro equipo) no requiere tocar el
código del agente en absoluto.

**Siguiente:** [`05_tools_estilo_jub.ipynb`](05_tools_estilo_jub.ipynb) — tools de
consulta más ricas, con una mini-DSL, al estilo de las que usa `jub-agent` de verdad.


In [5]:
# Limpieza
server_process.terminate()
server_process.wait(timeout=5)
print("Servidor detenido.")


Servidor detenido.
